# Part 2: Creating a Basic Agent

![Alt text](img/augLLMs.png)

Let's walk through the fundamentals of building a basic agent.

At a high level, agents aren't magical — they are just LLMs augmented with the ability to call external tools when needed.  
In fact, a very simple agent is often just **an LLM in a while loop**, deciding when to call tools based on what the user asks.

The key concepts we'll cover:
- **Function Definition**: What tools the LLM can access.
- **Function Call**: A special output from the model indicating that a tool should be used.
- **Function Response**: The result we get from executing that tool.

To tie all of this together, we'll build a minimal **LLM Framework** that wraps around the model's outputs.

We'll also relate this to a broader view:  
In more complex systems, agents often combine **retrieval**, **tools**, and **memory** — but at the core, the pattern remains the same:  
The LLM outputs a call, and the framework decides what to do next.

**Original workshop recording:** The original workshop used Gemma 3; this notebook has been updated to Gemma 4. [Watch this section on YouTube.](https://www.youtube.com/live/-IWstEStqok?t=2088)

![functioncalling](img/function-calling-overview.png)

## Our First LLM Tool

Before we build our agent, we need at least one tool that the LLM can call.

This is a simple (and arbitrary) tool — but the same pattern applies to any function you might want the model to use.

We'll start by building a very basic weather lookup function, because it's intuitive and lets us focus on the function call mechanics rather than the tool itself.

In [1]:
import logging
import re
import json

def get_weather(city):
    logging.info("Called weather function %s", city)
    if city.lower() in {"austin", "sydney"}:
        return "sunny"
    else:
        return "cloudy"

In [2]:
get_weather("Austin")

'sunny'

## Sending a Basic Prompt to the Model

Now that we have a tool ready, let's set up a simple function to interact with the model.

We'll define a basic `model_call(prompt)` function that sends user input to the LLM and receives a response.  
This will be the foundation we build on as we add more complex behaviors like tool calling.

In [3]:
from ollama import chat
from ollama import ChatResponse

model = 'gemma4:latest'

# Note, the argument model_prompt is specific here
def model_call(model_prompt):
    
    response: ChatResponse = chat(model=model, messages=[
      {
        'role': 'user',
        'content': model_prompt,
      },
    ])
    return response['message']['content']

user_prompt = "Say hello to the class"

# Note, the argument user_prompt is specific here
model_call(user_prompt)

"Hello, class! 👋 I'm excited to be here and ready to help you learn anything you need. What amazing things are we going to explore today? 😊"

## Guiding the Model with a System Prompt

System prompts let us control how the model behaves before it sees the user's input.  
We'll define a simple function that prepends a system instruction to the user's message, allowing us to guide tone, style, or available tools.


In [4]:
def augmented_model_call(system_prompt, user_prompt, print_prompt=False):
    if print_prompt:
        print(f"[system]: {system_prompt}\n[user]: {user_prompt}")

    response: ChatResponse = chat(model=model, messages=[
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ])
    return response['message']['content']

In [5]:
# Example system prompt: talk like a pirate
# This is injected by the LLM application behind the scenes

system_prompt = '''Talk like a pirate to everyone. \n '''
user_prompt = "Say hello to the class"

augmented_model_call(system_prompt, user_prompt, print_prompt=True)

[system]: Talk like a pirate to everyone. 
 
[user]: Say hello to the class


'**AHOY THERE, YE SCURVY DOGS!** 🏴\u200d☠️\n\nHark, listen close to the booming voice of this old salt! May the tide favor ye all and yer wits be sharper than a freshly whetted cutlass!\n\nI bid ye a hearty, grand **HELLO!** I be ready to sail through whatever lessons ye have charted for this day, matey! Let the adventure begin!\n\n**ARRRRR!** ⚓️'

## Teaching the Model to Suggest Tool Calls

Now that we've seen how system prompts can guide behavior, we'll write a system prompt that teaches the model how to suggest when a tool should be used.  
We'll structure the prompt so that the model outputs a JSON instruction when it decides a function call is needed.

In [6]:
system_prompt = '''
You have the following functions available
 def get_weather(city: str)
   """Given a city returns the weather for that city"""

 If you call this function return only raw JSON with no markdown or backticks: [{"city": city}]
 otherwise respond normally
'''

In [7]:
user_prompt = "What's the weather in Sydney?"
augmented_model_call(system_prompt, user_prompt)

'{"city": "Sydney"}'

In [8]:
user_prompt = "How's the Austin forecast looking today?"
augmented_model_call(system_prompt, user_prompt)

'[{"city": "Austin"}]'

In [9]:
user_prompt = "How's the London weather looking today?"
augmented_model_call(system_prompt, user_prompt)

'{"city": "London"}'

In [10]:
user_prompt = "Say hi to Hugo"
augmented_model_call(system_prompt, user_prompt)

'Hi Hugo!'

## Parsing the Response

Now that the model knows how to suggest tool calls, we need to build a small framework around it to actually process results.

The basic logic we want:

1. If there is **no function call**, simply return the model's response to the user.
2. If there **is a function call**:
   - a. Call the appropriate tool to get new information.
   - b. Either return the tool result directly, or reinject it back into the model to generate a better final response.

We'll start by writing a simple function to detect when the model outputs a tool call in JSON format.

In [11]:
import re
import json

pattern = r'\[\{.*?\}\]'

def parse_response(model_response):
    if tool_call := re.search(pattern, model_response, re.DOTALL):
        return json.loads(tool_call.group(0))[0]
    return None

model_response = '[{"city": "Austin"}]'
print(parse_response(model_response))

model_response = 'Hi Hugo!'
print(parse_response(model_response))

{'city': 'Austin'}
None


**A note on the regex pattern:** With Gemma 3, the model always wrapped JSON output in backtick fences (` ```json ... ``` `), so the old pattern had to strip those first. Gemma 4 returns cleaner output with no fences, so the pattern is simpler.

However, Gemma 4 is occasionally inconsistent about the outer array brackets. You may notice above that Sydney and London returned `{"city": "Sydney"}` rather than `[{"city": "Sydney"}]`. The current pattern only matches the array-wrapped form, so those cases return `None`. Worth keeping an eye on if the agent seems to ignore certain cities.

In [12]:
model_response = 'hi! Hugo'
print(parse_response(model_response))

None


## Putting It All Together: Building Our First Agent

Now that we've built all the individual pieces, let's combine them to create our first real agent.  
This agent will:
- Send a prompt to the model.
- Detect if a tool call is needed.
- Call the tool and get the result.
- Optionally feed the tool result back into the model for a better final answer.

In [13]:
def chat_interaction(user_prompt):
    system_prompt = '''
    You have the following functions available:

    def get_weather(city: str)
    """Given a city returns the weather for that city"""

    If you call this function return only raw JSON with no markdown or backticks: [{"city": city}]
    otherwise respond normally
    '''

    # Get a model response. Right now we don't know if it's a function call or chat response
    model_response = augmented_model_call(system_prompt, user_prompt)

    # Check if the response contains a function call
    function_call_json = parse_response(model_response)

    # If it's not a function call, return the response
    if not function_call_json:
        return model_response
    
    # Since we detected a function call, get the weather
    weather = get_weather(function_call_json["city"])

    # Reinject the result into the model for a better final response
    function_response_prompt = f"The weather in {function_call_json['city']} is {weather}, tell me the weather as if you were a pirate"

    model_response = model_call(function_response_prompt)
    return model_response

chat_interaction("Can you say hi to Hugo?")

'Hi Hugo! 👋'

In [14]:
chat_interaction("What's the weather today in London?")

''

In [15]:
chat_interaction("How are things looking in Austin?")

"Ahoy, matey! Gather 'round and lend yer ear, for I've spied the skies and seen what the winds blow!\n\n**Avast ye, for the heavens themselves shine bright!**\n\n'Tis a fine day o' plunderin'! The sun, ye see, be starin' down upon us like a doubloon gold! The sky be a canvas o' the deepest azure, free o' any cloud-beast that could spoil a man's jolly mood.\n\nIt be perfect for swabbin' the decks and chartin' a course to buried treasure! Don't ye need yer cutlass, just yer shades and a thirsty tankard!\n\n**Shiver me timbers! The weather be fair, by the kraken's beard!** Yarr!"

##  Recap: What We Learned

In this section, we built our first basic agent that can recognize when a tool call is needed and respond accordingly.

Here are the key ideas to remember:
- **Function calls aren't magic**: We rely on the model's "reasoning" to decide when to call a function, based on the system prompt and user input.
- **The model doesn't actually call functions itself**: It simply outputs a structured signal (like JSON) suggesting what should happen.  
- **The framework — your code — decides what to do**: It parses the model’s output, calls tools when needed, and can reinject the results for a better final answer.

This pattern — LLM suggests, framework acts — is the foundation for building more complex agents later.

## Your Turn: Connect to a Real API

Now that you've seen how we can define simple tools manually,  
let's try using a real external API to fetch live data.

Below is a starting point — your task is to plug this into an agent loop, just like we did earlier with our fake `get_weather` function.

In [16]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

Let's test it by fetching the current temperature for London!

In [17]:
LONDON_LATITUDE = 51.5074 # Approximate latitude for London
LONDON_LONGITUDE = -0.1278 # Approximate longitude for London (West is negative)
get_weather(LONDON_LATITUDE, LONDON_LONGITUDE)

8.6

## Tool Call versus Agents

![functioncalling](img/AlwaysHasBeen.jpg)
Source: https://huggingface.co/blog/tiny-agents


Agents have a lot of definitions.  
The simplest is that they're a model that can call tools inside a while loop.

The model suggests an action, but it's our code that decides what actually happens.

## Native Tool Calling with Gemma 4

So far we've been doing tool calling manually — prompting the model to emit JSON, then parsing it with regex.

Gemma 4 supports **native tool calling** via Ollama's `tools` parameter. Instead of parsing the model's text output, you define tools as structured schemas and Ollama handles the detection for you.

This is cleaner and more reliable — but understanding the manual approach first shows you exactly what the framework is doing under the hood.

In [18]:
import ollama
import requests

# City name to coordinates lookup
CITY_COORDS = {
    "london": (51.5074, -0.1278),
    "sydney": (-33.8688, 151.2093),
    "austin": (30.2672, -97.7431),
}

def get_weather_for_city(city: str) -> str:
    """Fetch real current temperature for a city by name."""
    lat, lon = CITY_COORDS.get(city.lower(), (51.5074, -0.1278))
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m"
    )
    temp = response.json()['current']['temperature_2m']
    return f"{temp}°C"

# Define the tool schema
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather_for_city',
            'description': 'Given a city name, returns the current temperature',
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {
                        'type': 'string',
                        'description': 'The name of the city',
                    },
                },
                'required': ['city'],
            },
        },
    }
]

def native_chat_interaction(user_prompt):
    response = ollama.chat(
        model='gemma4:latest',
        messages=[{'role': 'user', 'content': user_prompt}],
        tools=tools,
    )

    if response.message.tool_calls:
        for tool_call in response.message.tool_calls:
            city = tool_call.function.arguments['city']
            weather = get_weather_for_city(city)
            return f"The weather in {city} is {weather}"

    return response.message.content

print(native_chat_interaction("What's the weather in Sydney?"))
print(native_chat_interaction("Say hi to Hugo"))

The weather in Sydney is 22.0°C
Hi there! 👋


## Thinking Mode with Gemma 4

Gemma 4 supports a thinking mode where the model reasons through its response before producing a final answer. You enable it by adding `<|think|>` at the start of your system prompt. The internal reasoning comes back in `response.message.thinking`, separate from the final response in `response.message.content`.

In [ ]:
system_prompt_thinking = '''<|think|>
You have the following functions available:

 def get_weather(city: str)
   """Given a city returns the weather for that city"""

 If you call this function return only raw JSON with no markdown or backticks: [{"city": city}]
 otherwise respond normally
'''

response = chat(model=model, messages=[
    {'role': 'system', 'content': system_prompt_thinking},
    {'role': 'user', 'content': "What's the weather in Sydney?"},
])

print("Thinking:", response.message.thinking)
print("Response:", response.message.content)

## Next Up: Part 3 — Discoverable Tools with MCP

In Part 3, we extend this agent to use the **Model Context Protocol (MCP)** — letting the model discover tools dynamically at runtime rather than from a hardcoded list.

The code for Part 3 lives in the `apps/` directory. Pick up the workshop recording right where we left off:

[Watch this section on YouTube](https://www.youtube.com/live/-IWstEStqok?t=5180)